# Reporte Técnico de Integración — AffiniScore

Este reporte analiza la integración y trazabilidad entre los módulos **frontend** (Ionic/Angular) y **backend** (FastAPI) del proyecto AffiniScore, bajo una arquitectura mono-repo.

## 1. Arquitectura de Integración (Frontend-Backend)

La comunicación entre el frontend (Ionic/Angular) y el backend (FastAPI) se realiza mediante peticiones HTTP (REST), utilizando endpoints bien definidos. El frontend consume la API a través de servicios Angular, gestionando el estado de carga y error para ofrecer una experiencia de usuario fluida.

### Rutas principales de la API

| Endpoint                                 | Método | Propósito                                 |
|-------------------------------------------|--------|--------------------------------------------|
| `/api/v1/partnerships/invite`            | POST   | Generar token de invitación                |
| `/api/v1/partnerships/join`              | POST   | Unirse a una relación con token            |
| `/api/v1/user_profiles`                   | GET    | Obtener perfil de usuario                  |
| `/api/v1/activities/catalog`             | GET    | Obtener catálogo de actividades            |

### Ejemplo de consumo desde Angular

```typescript
// Servicio Angular (supabase.service.ts)
async invitePartner(user1_id: string) {
  return this.http.post('/api/v1/partnerships/invite', { user1_id });
}

// Componente
this.supabaseService.invitePartner(this.userId)
  .subscribe({
    next: (resp) => { /* manejar éxito */ },
    error: (err) => { /* mostrar error */ }
  });
```

La gestión de estados de carga y error se realiza mediante variables reactivas (`isLoading`, `errorMessage`) y feedback visual en la UI (spinners, alertas).

## 2. Backend (FastAPI)

El backend implementa la lógica de negocio y expone endpoints REST. Utiliza validaciones de datos con Pydantic y maneja la seguridad mediante Supabase y Row Level Security (RLS).

### Ejemplo de endpoint relevante

```python
@app.post("/api/v1/partnerships/invite")
async def create_partnership(request: InviteRequest):
    token = ''.join(random.choices(string.ascii_uppercase + string.digits, k=6))
    response = supabase.table("partnerships").insert({
        "user1_id": request.user1_id,
        "pairing_token": token,
        "status": "pending"
    }).execute()
    return {"message": "Invitación creada", "token": token}
```

### Validaciones y manejo de errores
- Validación de datos de entrada con Pydantic.
- Manejo de errores con `HTTPException`.
- Seguridad: Uso de `service_role_key` para operaciones privilegiadas y RLS en Supabase para restringir el acceso a datos sensibles.

### Ejemplo de manejo de error
```python
if not response.data:
    raise HTTPException(status_code=400, detail="Token no válido")
```

## 3. Frontend (Ionic/Angular)

El frontend está estructurado en módulos y componentes Angular, utilizando servicios para la gestión del estado y la comunicación con la API.

### Estructura de componentes y servicios
- **Componentes:** Encapsulan la UI y gestionan la interacción del usuario (ej: registro, vinculación de pareja, dashboard).
- **Servicios:** Centralizan la lógica de negocio del cliente y el acceso a la API (ej: `SupabaseService`).

### Ejemplo de flujo de usuario: Vinculación de pareja
```typescript
// Componente
onInvite() {
  this.isLoading = true;
  this.supabaseService.invitePartner(this.userId)
    .subscribe({
      next: (resp) => {
        this.token = resp.token;
        this.isLoading = false;
      },
      error: (err) => {
        this.errorMessage = err.message;
        this.isLoading = false;
      }
    });
}
```

La UI se actualiza automáticamente según el estado de la petición (carga, éxito, error) usando variables reactivas.

## 4. Flujo de Datos End-to-End (Trazabilidad)

A continuación se describe el flujo completo para la acción "Generar código de pareja":

1. **Acción en UI:** El usuario pulsa "Generar código" en el componente de vinculación.
2. **Servicio Angular:** El componente llama a `invitePartner()` del servicio.
3. **Petición HTTP:** El servicio realiza un POST a `/api/v1/partnerships/invite`.
4. **Endpoint FastAPI:** El backend valida, genera el token y lo almacena en la base de datos.
5. **Interacción con Base de Datos:** Se inserta un registro en la tabla `partnerships` con estado `pending`.
6. **Respuesta:** El backend responde con el token generado.
7. **Actualización de UI:** El frontend muestra el token al usuario y actualiza el estado.

### Diagrama de flujo (pseudocódigo)

```mermaid
graph TD;
  UI[Usuario pulsa "Generar código"] --> Service[Servicio Angular llama API]
  Service --> HTTP[POST /api/v1/partnerships/invite]
  HTTP --> Backend[FastAPI valida y procesa]
  Backend --> DB[Inserta en partnerships]
  DB --> Backend
  Backend --> Service[Devuelve token]
  Service --> UI[Muestra token en UI]
```

## 5. Seguridad y Autenticación

La autenticación se basa en Supabase Auth (JWT). El frontend almacena el token de sesión de forma segura (en memoria o almacenamiento seguro del dispositivo) y lo envía en cada petición.

- **Frontend:**
  - Maneja el login, registro y refresco de sesión usando el SDK de Supabase.
  - El token se adjunta automáticamente en las peticiones a la API.

- **Backend:**
  - Valida el token recibido en cada endpoint.
  - Usa la `service_role_key` solo en el backend para operaciones privilegiadas.
  - Aplica Row Level Security (RLS) en Supabase para restringir el acceso a los datos según el usuario autenticado.

### Ejemplo de flujo de autenticación
```typescript
// Login en frontend
const { user, session, error } = await this.supabase.auth.signInWithPassword({ email, password });
// El token se usa en cada petición subsiguiente
```

```python
# Validación en backend (ejemplo conceptual)
from fastapi import Depends, HTTPException
from fastapi.security import HTTPBearer

security = HTTPBearer()

@app.get("/api/v1/user_profiles")
async def get_profile(credentials=Depends(security)):
    # Validar JWT y extraer user_id
    ...
```

## 6. Roadmap Técnico y Dependencias

### Estado de los módulos principales
- **Autenticación:** Completo (login, registro, refresco de sesión)
- **Gestión de usuarios:** Completo (creación y consulta de perfiles)
- **Vinculación de pareja:** Completo (generación y unión por código)
- **Catálogo de actividades y retos:** Completo
- **Sistema de puntos y log de acciones:** Completo
- **Chat en tiempo real:** En desarrollo
- **Notificaciones push:** Pendiente
- **Gamificación avanzada:** Pendiente

### Dependencias y conexión front-back
- El frontend depende de los endpoints REST del backend para todas las operaciones críticas.
- El backend depende de Supabase para autenticación, almacenamiento y RLS.

### Tabla comparativa: Módulo del Frontend vs Endpoint del Backend

| Módulo Frontend                | Endpoint Backend                        |
|-------------------------------|-----------------------------------------|
| Registro de usuario           | `/api/v1/auth/register` (Supabase Auth) |
| Login                         | `/api/v1/auth/login` (Supabase Auth)    |
| Perfil de usuario             | `/api/v1/user_profiles`                 |
| Vinculación de pareja         | `/api/v1/partnerships/invite`           |
| Unión por código              | `/api/v1/partnerships/join`             |
| Catálogo de actividades       | `/api/v1/activities/catalog`            |
| Canje de puntos               | `/api/v1/points/redeem`                 |
| Chat                          | `/api/v1/chat` (en desarrollo)          |